# Standalone MLE-STAR benchmark runner

This notebook runs paired offline OOF comparisons. It never submits by default. The catalog lists ten historical Kaggle tasks; the current executable training adapter is Leaf Classification, while the other modality adapters remain explicit implementation work rather than fabricated results.

In [ ]:
REPOSITORY = 'https://github.com/Isso-W/Jiaozi.git'
BRANCH = 'codex/mlestar-kaggle-benchmarks'
!rm -rf /content/mlestar-dataops
!git clone --depth 1 --branch {BRANCH} {REPOSITORY} /content/mlestar-dataops
%cd /content/mlestar-dataops
!python -m pip install -q '.[vision,llm,kaggle,dev]'
!python -m mlestar.cli benchmarks

In [ ]:
# Optional: add exactly one KAGGLE_API_TOKEN secret. This cell never prints it.
import os
try:
    from google.colab import userdata
    token = userdata.get('KAGGLE_API_TOKEN')
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token
except ImportError:
    pass

# Accept each competition's rules in the Kaggle UI before downloading its data.
SUBMIT = False

In [ ]:
# Fetch Leaf Classification data into DATA_ROOT if it is not already there.
# Uses the KAGGLE_API_TOKEN secret from the previous cell. The kaggle CLI
# reads KAGGLE_API_TOKEN directly from the environment -- no credentials
# file needs to be written here.
# Accept the competition rules on kaggle.com before running this cell, or
# skip it and upload train.csv/test.csv to DATA_ROOT yourself.
import os

DATA_ROOT = '/content/leaf-classification'
os.makedirs(DATA_ROOT, exist_ok=True)

if not os.path.exists(f'{DATA_ROOT}/train.csv'):
    if not os.environ.get('KAGGLE_API_TOKEN'):
        raise RuntimeError(
            'No train.csv in DATA_ROOT and no KAGGLE_API_TOKEN secret set. '
            'Either set the secret in the previous cell or upload train.csv/test.csv '
            'to DATA_ROOT yourself.'
        )
    !kaggle competitions download -c leaf-classification -p {DATA_ROOT}
    !cd {DATA_ROOT} && unzip -o leaf-classification.zip
    # Shell magics don't raise on failure, so verify the outcome ourselves.
    # If this fires, scroll up to the raw kaggle/unzip output above for the
    # real cause -- almost always unaccepted competition rules (visit
    # https://www.kaggle.com/c/leaf-classification/rules and click Join/
    # Accept) or an invalid/expired KAGGLE_API_TOKEN secret.
    if not os.path.exists(f'{DATA_ROOT}/train.csv'):
        raise RuntimeError(
            'Download did not produce train.csv in DATA_ROOT. Scroll up to the '
            'kaggle/unzip output above for the real error -- commonly the '
            'competition rules are not accepted yet at '
            'https://www.kaggle.com/c/leaf-classification/rules, or '
            'KAGGLE_API_TOKEN is invalid/expired.'
        )


In [ ]:
# Leaf Classification is the first fully executable adapter.
RUN_ROOT = '/content/mlestar-runs/leaf-classification'
!python -m mlestar.cli compare --benchmark leaf_classification --data-root {DATA_ROOT} --run-root {RUN_ROOT} --seeds 13 29 47 --no-submit

import pandas as pd
pd.read_csv(f'{RUN_ROOT}/comparison.csv')


A public leaderboard score is not a replacement for OOF comparison. The listed competitions are historical and expected to reject new submissions; record Kaggle's actual response if and only if you explicitly set `SUBMIT = True` after reviewing the competition rules and local submission schema.